# Heel-video shoe tracker — SAM 2 validation

This notebook answers one question cheaply: **when you twist your foot to show the shoe, does SAM 2 keep a clean outline on the shoe the whole time?**

If it does, replacing the shoe by compositing is realistic. If the outline slips or falls apart during the twist, that tells you the twist beat needs the heavier 3D approach.

**How to run it**
1. `Runtime` menu -> `Change runtime type` -> set **Hardware accelerator: GPU**, then Save.
2. Run each cell top to bottom (Shift+Enter).
3. When asked, upload **one short clip** (a few seconds is plenty for the first test).
4. Read the shoe's x,y off the first frame and type it in.
5. Watch the preview video at the end.

Nothing here is saved anywhere but this Colab session.

## 1. Confirm you have a GPU
If this errors or shows nothing, go back and set the runtime to GPU.

In [ ]:
!nvidia-smi

## 2. Install SAM 2 and download the model
Takes a couple of minutes the first time.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
print("Done. Model downloaded.")

## 3. Upload one of your clips
Run this, then pick a video file from your computer. Keep the first test short.

In [ ]:
import os, glob
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = "/content/sam2/" + list(uploaded.keys())[0]
print("Uploaded:", VIDEO_PATH)

## 4. Break the video into frames
SAM 2's video mode reads a folder of numbered images. This also prints how many frames you got.

In [ ]:
import os, glob
FRAME_DIR = "/content/frames"
os.system(f"rm -rf {FRAME_DIR}")
os.makedirs(FRAME_DIR, exist_ok=True)

# -q:v 2 = high quality JPEGs; frames named 00000.jpg, 00001.jpg, ...
os.system(f'ffmpeg -loglevel error -i "{VIDEO_PATH}" -q:v 2 -start_number 0 "{FRAME_DIR}/%05d.jpg"')

frames = sorted(glob.glob(f"{FRAME_DIR}/*.jpg"))
print(f"Extracted {len(frames)} frames into {FRAME_DIR}")

## 5. Find the shoe in the first frame

Run this to show frame 0 with a coordinate grid. Look at where the **shoe** is and note an (x, y) pixel that sits squarely on it — x runs left-to-right, y runs top-to-bottom.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img0 = Image.open(frames[0])
w, h = img0.size
fig, ax = plt.subplots(figsize=(12, 12 * h / w))
ax.imshow(img0)
ax.set_xticks(range(0, w, max(1, w // 20)))
ax.set_yticks(range(0, h, max(1, h // 20)))
ax.grid(color="cyan", linestyle=":", linewidth=0.6, alpha=0.7)
ax.set_title(f"Frame 0  —  image is {w} wide x {h} tall")
plt.show()

## 6. Point at the shoe and preview the first-frame mask

Set `SHOE_X` / `SHOE_Y` to the pixel you read off above. Optionally add **negative** points (things to exclude, like the floor or the other foot) if the mask grabs too much.

This shows you the mask on frame 0 *before* tracking the whole video, so you can adjust the click until the outline looks right. Re-run this cell as many times as you like.

In [ ]:
import numpy as np
import torch
from sam2.build_sam import build_sam2_video_predictor

# ---- edit these two numbers ----
SHOE_X = 640
SHOE_Y = 900
# ---- optional: points to EXCLUDE, e.g. [[300, 950], [1000, 980]] ----
NEGATIVE_POINTS = []
# ---------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    "checkpoints/sam2.1_hiera_large.pt",
    device=device,
)

state = predictor.init_state(video_path=FRAME_DIR)
predictor.reset_state(state)

pts = [[SHOE_X, SHOE_Y]] + list(NEGATIVE_POINTS)
labels = [1] + [0] * len(NEGATIVE_POINTS)   # 1 = keep, 0 = exclude

_, obj_ids, mask_logits = predictor.add_new_points_or_box(
    inference_state=state, frame_idx=0, obj_id=1,
    points=np.array(pts, dtype=np.float32),
    labels=np.array(labels, dtype=np.int32),
)

mask0 = (mask_logits[0] > 0).cpu().numpy().squeeze()

fig, ax = plt.subplots(figsize=(12, 12 * h / w))
ax.imshow(img0)
overlay = np.zeros((*mask0.shape, 4))
overlay[mask0] = [1, 0, 0, 0.5]   # red where the mask is
ax.imshow(overlay)
ax.plot(SHOE_X, SHOE_Y, "yo", markersize=10)
for nx, ny in NEGATIVE_POINTS:
    ax.plot(nx, ny, "x", color="white", markersize=10)
ax.set_title("Red = what SAM 2 thinks the shoe is. Adjust the click if this is wrong.")
plt.show()

## 7. Track the shoe through the whole clip

Once frame 0 looks right above, run this. It follows the shoe across every frame.

In [ ]:
masks = {}
for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
    masks[frame_idx] = (mask_logits[0] > 0).cpu().numpy().squeeze()
print(f"Tracked the shoe across {len(masks)} frames.")

## 8. Build the preview video

This tints the tracked shoe red in every frame and stitches an MP4. **Watch the twist**: the red should stay glued to the shoe as the foot rotates.

In [ ]:
import cv2

out_frames_dir = "/content/overlay_frames"
os.system(f"rm -rf {out_frames_dir}")
os.makedirs(out_frames_dir, exist_ok=True)

for i, fpath in enumerate(frames):
    frame = cv2.imread(fpath)                # BGR
    m = masks.get(i)
    if m is not None and m.any():
        red = np.zeros_like(frame); red[:, :, 2] = 255
        frame = np.where(m[..., None], (0.5 * frame + 0.5 * red).astype(np.uint8), frame)
    cv2.imwrite(f"{out_frames_dir}/{i:05d}.jpg", frame)

os.system(f'ffmpeg -loglevel error -y -framerate 24 -i "{out_frames_dir}/%05d.jpg" '
          f'-c:v libx264 -pix_fmt yuv420p /content/preview.mp4')
print("Wrote /content/preview.mp4")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

data = b64encode(open("/content/preview.mp4", "rb").read()).decode()
HTML(f'<video width=500 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## How to read the result

- **Red stays locked on the shoe through the twist** -> compositing is viable. The mask you just produced is exactly the region you'd paint a new shoe into.
- **Red slips onto the floor, shrinks, or flickers during the twist** -> the twist beat is the hard part and will likely need the 3D-rendering approach; the walk-in/out can still use this.

Either way, that one twist moment is the whole ballgame. Once you've watched it, tell me what happened and we'll decide the next step.

*Tip:* if tracking is shaky, go back to cell 6, nudge the click, add negative points on the floor/other foot, and re-run 6 -> 7 -> 8.